# Multipart RVQ Residual Recoverability Audit

This pilot fixes the current infiller's predicted q0 and measures how much q1-q3 can repair it. The target is the cumulative latent represented by the ground-truth multipart codec tokens, so the experiment isolates infilling recoverability from the codec reconstruction floor.

Compared token constructions:

- `codec_gt`: all ground-truth codec tokens.
- `one_shot`: current `generate_steps=1` output.
- `pred_q0_gt_tail`: predicted q0 plus the original ground-truth q1-q3.
- `pred_q0_greedy_tail`: predicted q0 plus greedy residual requantization.
- `pred_q0_beam_tail`: predicted q0 plus beam-search residual requantization.


In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

here = Path.cwd().resolve()
PROJECT_ROOT = next(
    candidate for candidate in (here, *here.parents)
    if (candidate / 'motion_generation').is_dir()
)
MOTION_GENERATION_DIR = PROJECT_ROOT / 'motion_generation'
for path in (PROJECT_ROOT, MOTION_GENERATION_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from utils.multipart_motion import PART_ORDER
from utils.msd.complexity_error_audit import (
    default_audit_config,
    load_audit_context,
    path_status,
)
from utils.msd.residual_recoverability_audit import (
    paired_recovery_summary,
    plot_recovery_by_part,
    plot_recovery_fraction,
    run_residual_recoverability_audit,
    save_recovery_tables,
)

print('Project root:', PROJECT_ROOT)


## Configuration

`MP_STEP2_CKPT` must point to the multipart **Step 2 transformer** directory containing `config.json` and `model.safetensors` or `pytorch_model.bin`. It must not point to the parent multipart RVQ-VAE directory. The four codec paths point to individual `best.pth` files.


In [ ]:
config = default_audit_config(PROJECT_ROOT)
config.mask_checkpoint = Path(os.environ.get(
    'MP_STEP2_CKPT',
    str(PROJECT_ROOT / 'checkpoints' / 'mask_multipart'),
)).expanduser()

for part in PART_ORDER:
    env_name = f'MP_{part.upper()}_CKPT'
    if env_name in os.environ:
        config.part_checkpoints[part] = Path(os.environ[env_name]).expanduser()

config.device = os.environ.get('AUDIT_DEVICE', 'cuda:0')
config.max_clips = 64          # Pilot: evenly sampled validation clips
config.max_windows = 1024      # Pilot: evenly sampled windows
config.batch_size = 32
config.generate_steps = 1      # Current one-shot baseline
config.include_decoded_errors = True
config.include_fk_speed = False
config.output_dir = (
    PROJECT_ROOT / 'motion_generation' / 'outputs' / 'residual_recoverability_audit'
)
BEAM_WIDTH = 32

display(path_status(config))
print('Output:', config.output_dir)
print('Beam width:', BEAM_WIDTH)


Run the pilot first. Increase `max_clips` and `max_windows`, or set them to `None`, only after memory use and runtime look reasonable. Beam width 32 is an approximate recovery bound; greedy recovery is always reported separately.


In [ ]:
context = load_audit_context(config)
print('Selected windows:', len(context.selected_indices))
print('Loaded clips:', len(context.sequences))
print('Parts:', context.decoder.part_order if context.decoder is not None else None)


In [ ]:
tables = run_residual_recoverability_audit(context, beam_width=BEAM_WIDTH)
save_recovery_tables(tables, config.output_dir)
display(pd.DataFrame([tables.metadata]))
display(tables.failures.head(20))


## First checks

The main decision should use samples where q0 is wrong. Correct-q0 samples are retained as a sanity check: adaptive tails should reconstruct the target nearly exactly there.


In [ ]:
q0_accuracy = (
    tables.tokens.groupby('part', as_index=False)
    .agg(samples=('name', 'size'), q0_accuracy=('q0_correct', 'mean'))
)
display(q0_accuracy.round(5))
display(tables.summary.round(5))


In [ ]:
paired = paired_recovery_summary(tables.recovery, q0_errors_only=True)
display(paired.round(5))

primary = paired.loc[
    paired['candidate'].eq('pred_q0_beam_tail')
    & paired['reference'].eq('pred_q0_gt_tail')
    & paired['metric'].isin(['latent_rmse', 'decoded_rmse'])
]
print('Primary recoverability result on incorrect q0:')
display(primary.round(5))


In [ ]:
fig = plot_recovery_by_part(
    tables.recovery, metric='latent_rmse', reference='pred_q0_gt_tail'
)
plt.show()

fig = plot_recovery_by_part(
    tables.recovery, metric='decoded_rmse', reference='one_shot'
)
plt.show()

fig = plot_recovery_fraction(
    tables.recovery, metric='latent_rmse', candidate='pred_q0_beam_tail'
)
plt.show()


## How to decide

Focus on `pred_q0_beam_tail` for incorrect-q0 samples:

- Positive `mean_gain` means later codebooks can compensate for q0.
- `median_recovery_fraction = 0.30` means the adaptive tail removes 30% of the fixed-q0 error.
- High `improved_fraction` means the effect is consistent rather than driven by a few windows.
- A strong latent gain but weak decoded gain means the decoder is insensitive to that latent correction.
- Little beam gain over greedy means inexpensive greedy adaptive targets are sufficient.
- Little recovery from either method means q0 needs stronger prediction or top-k/beam handling; self-forcing later quantizers cannot repair it.

Do not interpret token accuracy as the primary outcome here. Adaptive residual codes are allowed to differ from the original q1-q3 while reconstructing a closer cumulative latent.


In [ ]:
tail_change_rows = []
for part, group in tables.tokens.groupby('part'):
    row = {'part': part, 'samples': len(group)}
    for q in range(1, context.decoder.num_quantizers):
        row[f'greedy_q{q}_change_rate'] = (group[f'greedy_q{q}'] != group[f'gt_q{q}']).mean()
        if f'beam_q{q}' in group:
            row[f'beam_q{q}_change_rate'] = (group[f'beam_q{q}'] != group[f'gt_q{q}']).mean()
    tail_change_rows.append(row)
display(pd.DataFrame(tail_change_rows).round(5))
